In [8]:
# FULL TRAINING FLOW

data_manager = DataManager()
feature_engineer = FeatureEngineer()
preprocessor_manager = PreprocessorManager()
model_manager = ModelManager()

X_train, X_test, y_train, y_test = data_manager.prepare_train_test_data()

X_train = feature_engineer.engineer_features(X_train)
X_test = feature_engineer.engineer_features(X_test)

X_train_processed, X_test_processed, _ = preprocessor_manager.prepare_train_test(
    X_train, X_test
)

model, metrics = model_manager.train_and_evaluate(
    X_train_processed, y_train, X_test_processed, y_test
)

preprocessor_manager.save()
model_manager.save()

In [13]:
# FULL API FLOW

X_pred = pd.DataFrame([payload])

feature_engineer = FeatureEngineer()
preprocessor_manager = PreprocessorManager()
model_manager = ModelManager()

preprocessor_manager.load()
model_manager.load()

X_pred = feature_engineer.engineer_features(X_pred)
X_pred_processed = preprocessor_manager.transform(X_pred)

y_pred = model_manager.predict(X_pred_processed)
y_prob = model_manager.predict_proba(X_pred_processed)

KeyError: 'reserved_room_type'

In [1]:
# Check usage of DataManager()

from eda_package.registry import *
from eda_package.data import DataManager

data_manager = DataManager()
df = data_manager.prepare_dataset()
X_train, X_test, y_train, y_test = data_manager.prepare_train_test_data()

In [2]:
# Check usage of FeatureEngineer()

from eda_package.features import FeatureEngineer

feature_engineer = FeatureEngineer()
X_train = feature_engineer.engineer_features(X_train)
X_test = feature_engineer.engineer_features(X_test)

In [3]:
# Check usage of PreprocessorManager()

from eda_package.preprocessor import PreprocessorManager

preprocessor_manager = PreprocessorManager()

X_train_processed, X_test_processed, preprocessor = (
    preprocessor_manager.prepare_train_test(X_train, X_test)
)
preprocessor_manager.save()

In [4]:
# Check usage of ModelManager()

from eda_package.model import ModelManager

model_manager = ModelManager()

# Training
model_manager.train(X_train_processed, y_train)
metrics = model_manager.evaluate(X_test_processed, y_test)
model_manager.save()

In [ ]:
class SimpleModelPipeline:
    """
    End-to-end pipeline:
        1.  Load → clean → feature-engineer → temporal split
        2.  Train a calibrated logistic regression
        3.  Predict per-booking cancel probability
        4.  Aggregate by (date, room_type): expected cancellations,
            std dev, 95 % CI
        5.  For each group, sweep extra bookings 0 … 30 using the
            Poisson-Binomial distribution and pick the level that
            maximises  E[revenue] – E[relocation cost]  while keeping
            P(relocation) ≤ max_risk
        6.  Return a single DataFrame with everything
    """

    def __init__(
        self,
        path: str = "/Users/beza/code/bezawitzyb/noshowshield/raw_data/hotel_bookings.csv",
        country_limit: int = COUNTRY_LIMIT,
        split_year: int = SPLIT_YEAR,
        ordinal_features_map: dict = None,
        model_folder: str = "models",
        random_state: int = 42,
        relocation_cost: float = DEFAULT_RELOCATION_COST,
        max_risk: float = DEFAULT_MAX_RISK,
        max_extra_sweep: int = MAX_EXTRA_SWEEP,
    ):
        self.path = path
        self.country_limit = country_limit
        self.split_year = split_year
        self.ordinal_features_map = ordinal_features_map or ORDINAL_FEATURES_MAP
        self.model_folder = model_folder
        self.random_state = random_state
        self.relocation_cost = relocation_cost
        self.max_risk = max_risk
        self.max_extra_sweep = max_extra_sweep

        self.model = None
        self.test_df = None       # raw test rows (kept for grouping)
        self.capacity_map = None  # inferred {room_type: capacity}

    # ================================================================
    #  1.  DATA PIPELINE
    # ================================================================
    def _build_arrival_date(self, df: pd.DataFrame) -> pd.DataFrame:
        """Compose a datetime from the split year / month-name / day columns."""
        df = df.copy()
        df["arrival_date"] = pd.to_datetime(
            df["arrival_date_year"].astype(str)
            + "-"
            + df["arrival_date_month"]
            + "-"
            + df["arrival_date_day_of_month"].astype(str),
            format="%Y-%B-%d",
        )
        return df

    def _infer_capacity(self, df: pd.DataFrame) -> dict:
        """
        Infer physical room capacity per room type from the data.

        Strategy: for each room type, take the maximum number of
        bookings observed on any single arrival date.  This is a
        reasonable proxy when real capacity isn't provided.
        """
        counts = (
            df.groupby(["arrival_date", "assigned_room_type"])
            .size()
            .reset_index(name="n_bookings")
        )
        capacity = (
            counts.groupby("assigned_room_type")["n_bookings"]
            .max()
            .to_dict()
        )
        return capacity

    def load_and_preprocess(self):
        """Run the full data pipeline and return processed train/test arrays."""
        df = load_raw_data(self.path)
        df = clean_data(df)
        df = group_countries(df, self.country_limit)
        df = engineer_features(df)
        df = self._build_arrival_date(df)

        # Infer capacity from the FULL dataset (before splitting)
        self.capacity_map = self._infer_capacity(df)

        train, test = temporal_split(df, self.split_year)
        self.test_df = test.copy()

        X_train, y_train = split_X_y(train)
        X_test, y_test = split_X_y(test)

        # Drop columns that are only used for aggregation, not modelling
        drop_cols = ["arrival_date"]
        X_train = X_train.drop(columns=[c for c in drop_cols if c in X_train.columns])
        X_test = X_test.drop(columns=[c for c in drop_cols if c in X_test.columns])

        X_train_processed, X_test_processed, _ = preprocess_pipeline(
            X_train, X_test, self.ordinal_features_map
        )
        return X_train_processed, X_test_processed, y_train, y_test

    # ================================================================
    #  2.  MODEL TRAINING
    # ================================================================
    def train(self, X_train, y_train):
        base = LogisticRegression(
            max_iter=500,
            class_weight="balanced",
            random_state=self.random_state,
        )
        self.model = CalibratedClassifierCV(base, method="sigmoid", cv=3)
        self.model.fit(X_train, y_train)
        return self.model

    # ================================================================
    #  3.  PREDICTION
    # ================================================================
    def predict(self, X):
        if self.model is None:
            raise ValueError("Model is not trained yet.")
        return self.model.predict(X)

    def predict_proba(self, X):
        if self.model is None:
            raise ValueError("Model is not trained yet.")
        return self.model.predict_proba(X)[:, 1]

    # ================================================================
    #  4.  EVALUATION
    # ================================================================
    def evaluate(self, y_true, y_pred, y_proba, verbose=True):
        metrics = {
            "accuracy": accuracy_score(y_true, y_pred),
            "roc_auc": roc_auc_score(y_true, y_proba),
            "precision": precision_score(y_true, y_pred, pos_label=1),
            "recall": recall_score(y_true, y_pred, pos_label=1),
            "brier": brier_score_loss(y_true, y_proba),
        }
        targets = {"roc_auc": 0.85, "precision": 0.80, "recall": 0.75, "brier": 0.15}

        if verbose:
            print("\n── Evaluation ──")
            for k, v in metrics.items():
                line = f"  {k:<12}: {v:.4f}"
                if k in targets:
                    target = targets[k]
                    ok = (v >= target) if k != "brier" else (v <= target)
                    line += f"  (target {'≤' if k == 'brier' else '≥'} {target}  {'✅' if ok else '❌'})"
                print(line)
        return metrics

    # ================================================================
    #  5.  POISSON-BINOMIAL HELPERS
    # ================================================================
    @staticmethod
    def _poisson_binomial_pmf(probs: np.ndarray) -> np.ndarray:
        probs = np.asarray(probs, dtype=np.float64)
        n = len(probs)
        pmf = np.zeros(n + 1)
        pmf[0] = 1.0
        for p in probs:
            new = np.empty_like(pmf)
            new[0] = pmf[0] * (1 - p)
            new[1:] = pmf[1:] * (1 - p) + pmf[:-1] * p
            pmf = new
        return pmf

    @staticmethod
    def _poisson_binomial_stats(probs: np.ndarray):
        probs = np.asarray(probs, dtype=np.float64)
        mean = probs.sum()
        var = (probs * (1 - probs)).sum()
        return mean, var

    # ================================================================
    #  6.  OVERBOOKING OPTIMISER  (per group)
    # ================================================================
    def _optimise_group(
        self,
        cancel_probs: np.ndarray,
        capacity: int,
        mean_adr: float,
    ) -> dict:
        cancel_probs = np.asarray(cancel_probs, dtype=np.float64)
        n_current = len(cancel_probs)
        show_probs = 1.0 - cancel_probs

        best = {
            "recommended_extra": 0,
            "recommended_total": n_current,
            "net_benefit": 0.0,
            "additional_revenue": 0.0,
            "expected_relocation_cost": 0.0,
            "relocation_probability": 0.0,
        }

        for extra in range(0, self.max_extra_sweep + 1):
            total = n_current + extra

            if extra > 0:
                mean_cancel = cancel_probs.mean()
                extended_show = np.concatenate([
                    show_probs,
                    np.full(extra, 1.0 - mean_cancel),
                ])
            else:
                extended_show = show_probs

            show_pmf = self._poisson_binomial_pmf(extended_show)

            if capacity + 1 <= total:
                p_relocate = show_pmf[capacity + 1:].sum()
            else:
                p_relocate = 0.0

            if p_relocate > self.max_risk:
                break

            expected_excess = sum(
                (k - capacity) * show_pmf[k]
                for k in range(capacity + 1, total + 1)
            )

            revenue = extra * mean_adr
            cost = expected_excess * self.relocation_cost
            profit = revenue - cost

            if profit >= best["net_benefit"]:
                best = {
                    "recommended_extra": extra,
                    "recommended_total": total,
                    "net_benefit": round(profit, 2),
                    "additional_revenue": round(revenue, 2),
                    "expected_relocation_cost": round(cost, 2),
                    "relocation_probability": round(p_relocate, 4),
                }

        return best